In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import udf, col
import pyspark.sql.functions as F


In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%python
dbutils.widgets.text("1_storageName", "adlsmartdata0912")
dbutils.widgets.text("2_container", "bronze")
dbutils.widgets.text("3_catalog", "catalog_smartdata")
dbutils.widgets.text("4_schema", "bronze")
dbutils.widgets.text("5_schema", "silver")

In [0]:
%python
storage_name = dbutils.widgets.get("1_storageName")
container = dbutils.widgets.get("2_container")
catalog =  dbutils.widgets.get("3_catalog")
schema_origen =  dbutils.widgets.get("4_schema")
schema_destino =  dbutils.widgets.get("5_schema")

In [0]:
%python
table_cliente  = "cliente"
table_pasivo  = "pasivo"
table_activo    = "activo"

table_cliente_natural = "cliente_natural"
table_pasivo_producto = "pasivo_productos"
table_activo_producto = "activo_producto"

In [0]:
%python

df_cliente = spark.table(f"{catalog}.{schema_origen}.{table_cliente}");
df_pasivo =  spark.table(f"{catalog}.{schema_origen}.{table_pasivo}");
df_activo = spark.table(f"{catalog}.{schema_origen}.{table_activo}");
df_cliente.cache();
df_pasivo.cache();
df_activo.cache();

In [0]:
%python
df_cliente.display()

In [0]:
%python
def perfilingreso_cliente(ingresoanual):
    print(f"Monto: {ingresoanual}")
    if    ingresoanual <= 20000:
        return "PERSONAS"
    elif  ingresoanual <= 50000:
        return  "PREFERENCIAL"
    elif  ingresoanual <= 150000:
        return "PLUS"
    else:
        return "NO INGRESO"
     

In [0]:
%python
def estado_cliente(estadocivil):
    print(f"Monto: {estadocivil}")
    if    estadocivil == "SOL":
        return "SOLTERO"
    elif  estadocivil == "CAS":
        return  "CASADO"
    elif  estadocivil == "DIV":
        return "DIVORCIADO"
    elif  estadocivil == "VIU":
        return "VIUDO"
    else:
        return "OTROS"

In [0]:
%python
def provincia_cliente(provinciadomicilio):
    print(f"Monto: {provinciadomicilio}")
    if    provinciadomicilio == "1701":
        return "PICHINCHA"
    elif  provinciadomicilio == "1703":
        return  "TUNGURAHUA"
    elif  provinciadomicilio == "1709":
        return "GUAYAS"
    else:
        return "OTRAS"

In [0]:
%python
perfilingreso_cliente_udf = F.udf(perfilingreso_cliente, StringType())
estado_cliente_udf = F.udf(estado_cliente, StringType())
provincia_cliente_udf = F.udf(provincia_cliente, StringType())

In [0]:

%python
df_cli_limpio = (df_cliente.withColumn("Nombre", translate(col("Nombre"),"'","")) 
.withColumn("Provincia", translate (col("Provincia"),"'",""))
.withColumn("EstadoCivil", translate (col("EstadoCivil"),"'",""))
.withColumn("FechaNacimiento", translate (col("FechaNacimiento"),"'",""))
.withColumn("IngresoAnual", coalesce(col("IngresoAnual"), lit(0)))
)

df_cli_limpio.display();

In [0]:
%python

df_pasivo = df_pasivo.withColumnRenamed("ClienteId", "ClienteidP")
df_activo = df_activo.withColumnRenamed("ClienteId", "ClienteidA")

In [0]:
%python
df_pasivo.display()

In [0]:
%python


df_cli_pro = df_cli_limpio.join(df_pasivo, df_cli_limpio['ClienteId'] == df_pasivo['ClienteidP'], "left")\
                         .join(df_activo, df_cli_limpio['ClienteId'] == df_activo['ClienteidA'], "left");


df_cli_pro = (
             df_cli_pro.withColumn("PerfilIngreso", perfilingreso_cliente_udf (col("IngresoAnual")))
            .withColumn("EstadoCivil_Descripcion", estado_cliente_udf(col("EstadoCivil")))
            .withColumn("Provincia_Descripcion", provincia_cliente_udf (col("Provincia")))
            )

df_cli_pro=df_cli_pro.select("ClienteId","Nombre","FechaNacimiento","IngresoAnual","PasivoId","SaldoPasivo","ActivoId","SaldoActivo","PerfilIngreso","EstadoCivil_Descripcion","Provincia_Descripcion")

#df_cli_pro.display();

df_cli_pro.write.mode("overwrite").saveAsTable(f"{catalog}.{schema_destino}.Cliente_Producto")



In [0]:
%python
df_kpi_cliente_proact_estadocivil =  (df_cli_pro.filter(col("ActivoId").isNotNull())
                                .groupBy("ActivoId","EstadoCivil_Descripcion")
                                .agg(F.count("*").alias("CantidadClientes"))
                                .orderBy("ActivoId",desc("CantidadClientes"))
                                )

df_kpi_cliente_proact_estadocivil = df_kpi_cliente_proact_estadocivil.select(col("ActivoId").alias("Producto"),"EstadoCivil_Descripcion","CantidadClientes")

#df_kpi_cliente_proact_estadocivil.display()

df_kpi_cliente_proact_estadocivil.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog}.{schema_destino}.kpi_productoactivo_estado")   

In [0]:
%python
df_kpi_cliente_propas_estadocivil =  (df_cli_pro.filter(col("PasivoId").isNotNull())
                                .groupBy("PasivoId","EstadoCivil_Descripcion")
                                        .agg(F.count("*").alias("CantidadClientes"))
                                        .orderBy("PasivoId",desc("CantidadClientes"))
                                )  
df_kpi_cliente_propas_estadocivil=  df_kpi_cliente_propas_estadocivil.select(col("PasivoId").alias("Producto"),"EstadoCivil_Descripcion","CantidadClientes")

#df_kpi_cliente_propas_estadocivil.display()

df_kpi_cliente_propas_estadocivil.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog}.{schema_destino}.kpi_productopasivo_estado")    

In [0]:
%python
df_kpi_cliente_propas_ingreso = (df_cli_pro.groupBy("PasivoId","PerfilIngreso")
                                        .agg(F.count("*").alias("CantidadClientes"),
                                             F.sum("IngresoAnual").alias("TotalIngreso"))
                                        .orderBy("PasivoId",desc("CantidadClientes"))
                                )
#df_kpi_cliente_propas_ingreso.display()

df_kpi_cliente_propas_ingreso =  df_kpi_cliente_propas_ingreso.select(col("PasivoId").alias("Producto"),"PerfilIngreso","TotalIngreso","CantidadClientes")

#df_kpi_cliente_propas_ingreso.display()

df_kpi_cliente_propas_ingreso.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(f"{catalog}.{schema_destino}.kpi_cliente_propas_ingreso")   